# 2. Basics of Optimization

## Part A Continuous optimization

Let's have a duct of length $L$ rigidly backed. 
You want to create an anechoic termination, so you insert two blocks of materials, A and B. 
Their total thickness is fixed: $L = \ell_A + \ell_B$, where $\ell_A$ is thickness of material A and $\ell_B$ is thickness of material B. 

Your task is to find optimal $\ell_A$ and $\ell_B$ to maximize the absorption.

![Geometry](optA.png)


In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from optimization_utils import *

If the whole length $L$ would be filled with one chosen material, how would the reflection and absorption look like?

In [ ]:
matA = MATERIALS["acoustic_foam"]
matB = MATERIALS["melamine_foam"]

# 2. Compute reflection and absorption spectra
r_matA, abs_matA = compute_spectrum([L], [matA])
r_matB, abs_matB = compute_spectrum([L], [matB])

# 3. Plot results
plt.figure(figsize=(15, 5))
plt.subplot(121)
plt.plot(freqs, np.abs(r_matA), label=matA.name)
plt.plot(freqs, np.abs(r_matB), label=matB.name)
setup_r_axis()

plt.subplot(122)
plt.plot(freqs, abs_matA, label=matA.name)
plt.plot(freqs, abs_matB, label=matB.name)
setup_abs_axis()

plt.show()


Now the optimization itself. Play around with the definition of objective that is to be minimized.

Note: How many degrees of freedom are there actually? In fact, you are finding the optimal position $l_A \in [0, L]$ and $l_B = L - l_A$.

In [ ]:
# Define optimization objective to minimize
def objective(x):
    l_A = x[0]
    l_B = L - l_A
    reflection, absorption = compute_spectrum([l_A, l_B], [matA, matB])

    # try different options:
    obj = -np.mean(absorption)            # Minimize negative mean absorption
    # obj = np.mean(np.abs(reflection))     # Does minimizing the mean reflection coefficient give the same result?
    # obj = -np.mean(absorption[freqs<300]) # Minimize negative mean absorption but only for frequencies below 300 Hz
    # obj = -np.max(absorption)             # Minimize negative max absorption
    return obj

# 3. Optimize l_A in [0, L]
res = minimize(objective, x0 = [L / 2], method='L-BFGS-B', bounds=[(0.0, L)])
opt_lA = res.x[0]
opt_lB = L - opt_lA

print(f"Optimal lenA ({matA.name}): {opt_lA * 1000:.2f} mm")
print(f"Optimal lenB ({matB.name}): {opt_lB * 1000:.2f} mm")



In [ ]:
# Reflection and absorption for optimal case and single-material baselines
r_opt, abs_opt  = compute_spectrum([opt_lA, opt_lB], [matA, matB])
r_matA, abs_matA = compute_spectrum([L], [matA])
r_matB, abs_matB = compute_spectrum([L], [matB])

# Plot results
plt.figure(figsize=(15, 5))
plt.subplot(121)
plt.plot(freqs, np.abs(r_opt), 'k-', linewidth=2, label=f'Part A: optimized ({matA.name} + {matB.name})')
plt.plot(freqs, np.abs(r_matA), '--', label=f'Only {matA.name}')
plt.plot(freqs, np.abs(r_matB), '--', label=f'Only {matB.name}')
setup_r_axis()

plt.subplot(122)
plt.plot(freqs, abs_opt, 'k-', linewidth=2, label=f'Part A: optimized ({matA.name} + {matB.name})')
plt.plot(freqs, abs_matA, '--', label=f'Only {matA.name}')
plt.plot(freqs, abs_matB, '--', label=f'Only {matB.name}')
setup_abs_axis()
plt.show()